In [9]:
import requests
import json
import os
import time
from dotenv import load_dotenv
load_dotenv()

True

In [10]:
PROMPT_TASK_SETUP = """
1. Go to https://aistudio.google.com/apps/drive/1AkZAMZLLWEuPnHr-6ZODHMnTfsfVVx9c?showCode=true&showFileTree=true&showTreeView=true&showPreview=true&resourceKey=
2. Click on "Make the app fullscreen"
3. Wait until the app is display fullscreen.
4. DONE TASK
"""


PROMPT_TASK_CHAT = """
The app have been built. Please test all functionality on this app.
After testing, please give a feedback about the app.
You can't interact with the app by DOM. Please use the tool execute_drag_operation to use mouse to interact with the app. This is the only way to interact with the app.
"""

In [11]:
URL_TEST = "https://aistudio.google.com/apps/drive/1AkZAMZLLWEuPnHr-6ZODHMnTfsfVVx9c?showCode=true&showFileTree=true&showTreeView=true&showPreview=true&resourceKey="

TASK_SETUP = {
    "name": "Input prompt to build app",
    "prompt": PROMPT_TASK_SETUP,
    "max_steps": 20,
    "output_model_fields": None,
    "exclude_actions": [],
    "llm_provider": "google",
    "llm_model": "gemini-2.0-flash",
    "llm_temperature": 0.0,
    "enable_memory": False,
    "memory_interval": 10,
    "initial_actions": [
        {"open_tab": {"url": URL_TEST}}
    ],
    # "use_vision_for_planner": True,  
    # "planner_interval": 1,           
    # "is_planner_reasoning": True,
    # "planner_llm": {
    #     "provider": "google",       
    #     "model": "gemini-2.0-flash",
    #     "temperature": 0.0
    # },             
}


OUTPUT_MODEL_FIELDS_CHAT = {
    "type": "object",
    "properties": {
        "feature": {
            "type": "string",
            "description": "The feature being tested (e.g. Dashboard, Ad Campaign)"
        },
        "feature_status": {
            "type": "string",
            "description": "Test result status (working/not working/partially working)"
        },
        "detail_reason": {
            "type": "string", 
            "description": "Detailed explanation of the test result"
        }
    },
    "required": ["feature", "feature_status", "detail_reason"]
}

TASK_CHAT = {
    "name": "Functionality Test",
    "prompt": PROMPT_TASK_CHAT,
    "max_steps": 20,
    "output_model_fields": OUTPUT_MODEL_FIELDS_CHAT,
    "exclude_actions": [
        "search_google",
        "go_back",
        "save_pdf",
        "switch_tab",
        "close_tab",
        "extract_content",
        "send_keys",
        "get_dropdown_options",
        "select_dropdown_options",
        "drag_drop",
        "click_element",
        "click_element_by_text",
        "click_element_by_index",
        "wait",
        "scroll",
        "paste_from_clipboard",
        "call_user_simulator",
        "get_system_message",
        "click_the_send_button",
        "go_down_the_page"
    ],
    "llm_provider": "google",
    "llm_model": "gemini-2.0-flash",
    "llm_temperature": 0.0,
    "enable_memory": True,
    "memory_interval": 10,
    "initial_actions": [],
    "use_vision_for_planner": True,  
    "planner_interval": 1,           
    "is_planner_reasoning": True,
    "planner_llm": {
        "provider": "openai",       
        "model": "o4-mini-2025-04-16",
        "temperature": 0.0,
    },             
    "report_config": {
        "provider": "google",
        "model": "gemini-2.0-flash",
        "temperature": 0.2,
        "is_report_reasoning": False,
        "use_vision_for_report": False,
        "report_folder": "E:/official_DopikAI/ai-agent-tester/tests_api/demo_simple",
        "extend_report_system_message": """
        """
    },       
}

In [12]:
import pandas as pd
test_cases = pd.read_csv("C:/Users/anpro/Downloads/chatui_case.csv")
test_cases = test_cases.to_dict(orient="records")

In [13]:
def create_payload(case_id):
    """Create task payload with prompts populated from test cases"""
    # Find the case in the test_cases list
    case = next((case for case in test_cases if case["case_id"] == case_id), None)
    if case is None:
        raise ValueError(f"Case ID {case_id} not found in test cases")
    
    # Create a copy of TASK_ to avoid modifying the original
    task = TASK_CHAT.copy()
    
    task['report_config']['report_folder'] = "E:/official_DopikAI/ai-agent-tester/reports" + "/" + case_id
    
    task_setup = TASK_SETUP.copy()
    task_setup['prompt'] = PROMPT_TASK_SETUP.format(
        title=case['title'],
    )
    payload = {
        "tasks": [task_setup, task],
        "laminar_api_key": os.getenv("LAMINAR_API_KEY", ""),
        "laminar_base_url": os.getenv("LAMINAR_BASE_URL", ""),
        "laminar_http_port": int(os.getenv("LAMINAR_HTTP_PORT", "0") or 0),
        "laminar_grpc_port": int(os.getenv("LAMINAR_GRPC_PORT", "0") or 0),
        "session_id": f"test-session-id-{case_id}",
        "simulator_provider": "google",
        "simulator_model": "gemini-2.0-flash",
        "simulator_temperature": 0.0,
        "simulator_task": "",
        "custom_actions": [],
        "use_own_browser": True,
        
    }
    
    return payload

In [14]:
# API endpoint
API_BASE_URL = "http://localhost:8081"

def run_test_case(case_id):
    """Run a test case with the specified case ID and return the results"""
    payload = create_payload(case_id)
    
    response = requests.post(f"{API_BASE_URL}/tasks/run", json=payload)
    
    # Print response
    print(f"Status code: {response.status_code}")
    print(f"Response: {response.json()}")
    
    if response.status_code == 200:
        # Extract task ID
        task_id = response.json()['data']["message"].split(": ")[1]
        print(f"Task ID: {task_id}")
        
        # Poll for results
        return poll_results(task_id)
    else:
        print(f"Failed to start task: {response.text}")
        return None


def poll_results(task_id):
    """Poll the API for task results and return the data"""
    
    print("Polling for task results...")
    max_attempts = 100
    attempts = 0
    
    while attempts < max_attempts:
        attempts += 1
        response = requests.get(f"{API_BASE_URL}/tasks/{task_id}")
        
        if response.status_code == 200:
            data = response.json()['data']
            status = data.get("status")
            
            print(f"Task status: {status}")
            
            if status == "completed":
                print("Task completed!")
                # print("Results:")
                # print(json.dumps(data.get("results"), indent=2))
                
                # Check for simulator interactions
                # simulator_interactions = data.get("simulator_interactions", [])
                # if simulator_interactions:
                #     print("\nUser Simulator Interactions:")
                #     print(json.dumps(simulator_interactions, indent=2))
                return data
            elif status == "failed":
                print("Task failed!")
                print("Error:")
                print(data.get("error"))
                return data
            elif status == "cancelled":
                print("Task was cancelled")
                return data
        
        # Wait before polling again
        time.sleep(5)
    
    print("Max polling attempts reached. Task may still be running.")
    return None

In [15]:
result = run_test_case("CB_001")

Status code: 200
Response: {'data': {'message': 'Task started with ID: 71f9cbdf-f309-4c81-bffa-8bf33eee4455'}, 'message': 'Success'}
Task ID: 71f9cbdf-f309-4c81-bffa-8bf33eee4455
Polling for task results...
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: completed
Task completed!
